In [ ]:
[meta_mix_simulation]
# ADSP + UKB sketch meta simulation.
# Each sketch is subset to `subset_n` pseudo-samples (the "full panel"),
# then mixed by `ADSP_prop` into a total meta sample of `total_n`.
parameter: genofile = paths
# Driver = GTEx bim files (used only as the variant-alignment anchor, as in template 1)
parameter: cwd = path("output")
parameter: job_size = 5
parameter: walltime = "100h"
parameter: mem = "120G"
parameter: numThreads = 2 
parameter: n_causal = 1
parameter: h2g = 0.05
parameter: total_n = 2000
# total meta sample size (n_ukb + n_adsp)
parameter: ADSP_prop = 0.5
# proportion of the meta sample coming from ADSP (n_ukb = total_n - n_adsp)
parameter: subset_n = 2000
# full-panel size each sketch is subset to
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_h2g_{h2g}_meta_mix.rds'
task: trunk_workers = 4, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(matrixStats)

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    # -----------------------------------------------------------------------------
    # Helpers
    # -----------------------------------------------------------------------------
    calculate_sumstat <- function(X, Y) {
        if (is.null(X) || nrow(X) == 0 || ncol(X) == 0) {
            return(tibble(
                SNP = character(), Beta = numeric(), se = numeric(), Freq = numeric(),
                p = numeric(), z = numeric(), N = integer(),
                chrom = integer(), pos = integer(), A2 = character(), A1 = character(),
                original_z = numeric(),
                variants_id_original = character(), variant_id = character()
            ))
        }
        rr <- susieR::univariate_regression(X, Y)
        z  <- rr$betahat / rr$sebetahat
        tibble(
            SNP  = colnames(X),
            Beta = rr$betahat,
            se   = rr$sebetahat,
            Freq = colSums(X) / (2 * nrow(X)),
            p    = 2 * pnorm(abs(z), lower.tail = FALSE),
            z    = z,
            N    = nrow(X)
        ) %>%
            separate(SNP, into = c("chrom", "pos", "A2", "A1"), sep = ":", remove = FALSE) %>%
            mutate(
                chrom = as.numeric(str_remove(chrom, "^chr")),
                pos   = as.integer(pos),
                original_z = z,
                variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
                variant_id = variants_id_original
            )
    }

    # Draw a cohort of size n from a full pseudo-genotype panel and compute sumstats.
    simulate_cohort <- function(G_full, n, B_mat, h2g) {
        if (n == 0) {
            return(list(
                Xmat = G_full[0, , drop = FALSE], pheno = NULL,
                target_tb = calculate_sumstat(G_full[0, , drop = FALSE], numeric(0))
            ))
        }
        idx   <- sample.int(nrow(G_full), n)
        Xmat  <- G_full[idx, , drop = FALSE]
        pheno <- sim_multi_traits(G = Xmat, B = B_mat, h2g = h2g, is_h2g_total = FALSE)$P
        target_tb <- calculate_sumstat(Xmat, as.numeric(pheno[, 1]))
        list(Xmat = Xmat, pheno = pheno, target_tb = target_tb)
    }

    # ADSP pgen sketch: undo the U_MIN/U_MAX scaling used to store the sketch.
    recover_U <- function(U_scaled, Umin, Umax) {
        sweep(sweep(U_scaled, 2, 2, "/"), 2, (Umax - Umin), "*") +
            matrix(Umin, nrow(U_scaled), ncol(U_scaled), byrow = TRUE)
    }

    # Align a reference panel (with its U matrix) to a target, keeping variant order.
    uniform <- function(target, reference, reference_U = NULL, flip_cols = FALSE) {
        qc_data <- allele_qc(
            target_data  = target,
            ref_variants = reference,
            col_to_flip  = if (flip_cols) c("Beta", "z") else NULL
        )
        sumstat_flipped <- qc_data$target_data_qced %>% arrange(pos)
        kept_vars       <- sumstat_flipped$variant_id
        out <- list(sumstat = sumstat_flipped)
        if (!is.null(reference_U)) out$U <- reference_U[kept_vars, , drop = FALSE]
        out
    }

    filter_vars <- function(input, variants) {
        sumstat   <- input$sumstat %>% filter(variants_id_original %in% variants)
        kept_vars <- sumstat %>% pull(variant_id)
        out <- list(sumstat = sumstat)
        if (!is.null(input$U)) out$U <- input$U[kept_vars, , drop = FALSE]
        out
    }


    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
    
      idx <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
    
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
    
      colnames(sketch2) <- qced$variants_id_original
    
      list(
        sumstat = qced,
        sketch = sketch2,
        flipped_prop = length(flip_index) / ncol(sketch2)
      )
    }

    # -----------------------------------------------------------------------------
    # Region index i (anchor = GTEx TAD, as in template 1)
    # -----------------------------------------------------------------------------
    file <- ${_input:r}
    i <- file %>% str_remove("\\.bim$") %>% str_remove(".*/gtex_")

    gtex_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/gtex_", i)
    gtex_data <- plink_process(gtex_path)            # bim used only as alignment anchor

    UKB_path  <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/UKBB_", i)
    UKB_data  <- plink_process(UKB_path)

    # -----------------------------------------------------------------------------
    # UKB sketch:  U_UKB = X_UKB' W    (variants x B, raw allele coding)
    # -----------------------------------------------------------------------------
    W     <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/random_vector/result/random_matrix_10000.rds")
    U_UKB <- crossprod(UKB_data$bed, W)              # rownames = variant_id (chr:pos:A2:A1)

    # -----------------------------------------------------------------------------
    # ADSP sketch (pgen): read X + afreq, recover raw U (variants x samples)
    # -----------------------------------------------------------------------------
    prefix <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes//ADSP_sketch_trimed/result/ADSP_", i)
    pvar   <- paste0(prefix, ".pvar"); pgen <- paste0(prefix, ".pgen"); psam <- paste0(prefix, ".psam")

    all_variant_info <- read_tsv(pvar)
    variant_idx      <- seq_len(nrow(all_variant_info))
    variant_info     <- all_variant_info[variant_idx, , drop = FALSE]
    psam <- as.data.frame(read_tsv(psam, show_col_types = FALSE))
    colnames(psam)[1:2] <- c("FID", "IID")

    pgen_obj <- pgenlibr::NewPgen(pgen)
    X <- pgenlibr::ReadList(pgen_obj, variant_subset = variant_idx, meanimpute = FALSE)
    rownames(X) <- psam$IID
    colnames(X) <- variant_info$ID

    chrom_adsp <- all_variant_info$`#CHROM`[1]
    afreq_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/chr", chrom_adsp,
                         "/ADSP.R4.EUR.chr", chrom_adsp, ".afreq")
    afreq <- read_tsv(afreq_path)
    afreq_ordered <- afreq %>% slice(match(colnames(X), ID))
    umin <- afreq_ordered %>% pull(U_MIN)
    umax <- afreq_ordered %>% pull(U_MAX)

    colnames(variant_info) <- c("chrom", "pos", "original_id", "A2", "A1")
    U_ADSP_rec <- recover_U(X, umin, umax)
    U_ADSP     <- U_ADSP_rec                     # variants x samples, rownames = variant ids
    ADSP_data  <- list(bim = variant_info, U = U_ADSP)

    target_tb <- UKB_data$bim %>%
        mutate(
            variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
            variant_id           = variants_id_original
        )

    harm <- sketch_harmonize(
      target_data  = target_tb,
      ref_variants = ADSP_data$bim,
      sketch       = U_ADSP
    )

    # ---- Full-panel genotypes ----
    UKB_geno_full <- UKB_data$bed[, colnames(harm$sketch), drop = FALSE]
    ADSP_geno_full <- harm$sketch
    p <- ncol(UKB_geno_full)
    
    # ---- Parameters ----
    ntrait   <- 1
    ncausal  <- ${n_causal}
    h2g      <- ${h2g}
    total_n  <- ${total_n}
    ADSP_prop <- ${ADSP_prop}
    
    # ---- Target cohort sizes ----
    n_ADSP <- round(ADSP_prop * total_n)
    n_UKB <- total_n - n_ADSP
    
    if (n_UKB > nrow(UKB_geno_full)) {
      stop(sprintf("n_UKB=%d exceeds UKB panel (%d); lower total_n or ADSP_prop.",
                   n_UKB, nrow(UKB_geno_full)))
    }
    if (n_ADSP > nrow(ADSP_geno_full)) {
      stop(sprintf("n_ADSP=%d exceeds ADSP panel (%d); lower total_n or raise ADSP_prop.",
                   n_ADSP, nrow(ADSP_geno_full)))
    }
    if (n_UKB < 0 || n_ADSP < 0) {
      stop("Negative cohort size computed; check total_n and ADSP_prop.")
    }
    
    # ---- Shared causal architecture ----
    set.seed(as.integer(i))
    causal_index <- sample_causal_index(Xmat = UKB_geno_full, ncausal = ncausal)
    
    B_mat <- matrix(0, nrow = p, ncol = ntrait)
    B_mat[, 1] <- sim_beta_fix_variant(
      G = UKB_geno_full,
      causal_index = causal_index,
      is_h2g_total = FALSE
    )
    
    # ---- Simulate cohorts conditionally ----
    UKB_out <- simulate_cohort(UKB_geno_full, n_UKB, B_mat, h2g)
    ADSP_out <- simulate_cohort(ADSP_geno_full, n_ADSP, B_mat, h2g)
    
    UKB_target_tb <- UKB_out$target_tb
    ADSP_target_tb <- ADSP_out$target_tb
    
    # ---- Panel LD cache ----
    panel_ld_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_ADSP_Mix_simulation/panel_LD"
    dir.create(panel_ld_dir, recursive = TRUE, showWarnings = FALSE)
    panel_ld_path <- file.path(panel_ld_dir, paste0("UKB_ADSP_", i, ".panelLD.rds"))
    
    if (!file.exists(panel_ld_path)) {
      saveRDS(list(
        UKB_LD_full  = get_cormat(UKB_geno_full),
        ADSP_LD_full  = get_cormat(ADSP_geno_full),
        variant_id   = colnames(UKB_geno_full),
        n_UKB_panel  = nrow(UKB_geno_full),
        n_ADSP_panel  = nrow(ADSP_geno_full)
      ), panel_ld_path)
    }
    
    # ---- Output bundle ----
    dats <- list(
      UKB_target_tb = UKB_target_tb,
      ADSP_target_tb = ADSP_target_tb,
      UKB_Xmat_full = UKB_geno_full,
      ADSP_Xmat_full = ADSP_geno_full,
      panel_ld_path = panel_ld_path,
      true_vars = causal_index,
      n_UKB = n_UKB,
      n_ADSP = n_ADSP,
      total_n = total_n,
      ADSP_prop = ADSP_prop,
      input_path = UKB_path
    )
    
    index <- ${_index}
    ss_dir <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_ADSP_Mix_simulation/sumstat/h2g_", h2g,
                     "/causal_", ncausal, "/prop_", ADSP_prop)
    dir.create(ss_dir, recursive = TRUE, showWarnings = FALSE)
    
    write_tsv(UKB_target_tb, file.path(ss_dir, paste0("UKB_", index, ".sumstat.tsv")))
    write_tsv(ADSP_target_tb, file.path(ss_dir, paste0("ADSP_", index, ".sumstat.tsv")))
    
    data_out_path <- ${_output:r}
    dir.create(dirname(data_out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(dats, data_out_path)

In [ ]:
#!/bin/bash
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="ADSP_UKB_Mix_simulation"
job_dir="${base_dir}/${job_name}"

# total sample size (EUR + AFR); must be <= EUR panel size (~489)
total_n=2000

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 5
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/12.Fig4A_4B_S9A.ADSP_UKB_Meta.ipynb" meta_mix_simulation \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/*.bim) \
    --n_causal CAUSAL --h2g H2G --ADSP_prop PROP --total_n TOTAL_N --mem 60G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/prop_PROP/ -j 5
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== Generate + submit one job per (causal, h2g, afr_prop) =====
for causal in 1 2 3; do
  for h2g in 0.02 0.04 0.06; do
      for prop in 0.0 0.20 0.50 0.80 1.0; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|TOTAL_N|${total_n}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
      sb "${output_script}"   # use sb if you aliased
    done
  done
done

In [ ]:
[finemap_sweep]
# Fine-map the UKB+ADSP natural-weight meta z at one (h2g, causal, prop).
# prop = ADSP sample-size proportion; the realized weight is w = n_ADSP / total_n.
#   z_meta = (sqrt(n_ADSP) z_ADSP + sqrt(n_UKB) z_UKB) / sqrt(N)     [natural weight]
#   n_mix  = total_n                                              [n_eff(natural) = N]
# LD conditions:
#   *_insample : subsample LD (the exact simulating individuals) -> GOLD generative
#   *_refpanel : full-panel LD (from panel_ld_path)              -> realistic reference-panel
parameter: result_root = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_Mix_simulation/result")
parameter: cwd         = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_fine/result")

# sweep parameters
parameter: h2g    = "0.05"
parameter: causal = "1"
parameter: prop   = "0.66"

parameter: job_size = 30
parameter: walltime = "40h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""

# one job per LD sample file in this (h2g, causal, prop) folder
input: [str(x) for x in path(f"{result_root}/h2g_{h2g}/causal_{causal}/prop_{prop}").glob("sample_*_mix.rds")], group_by = 1

output: f'{cwd:a}/h2g_{h2g}/causal_{causal}/prop_{prop}/{_input[0]:bn}.rds'

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------


    # Local susieR build (multi-panel / R_finite) -- required for the list() reference
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)



    h2g    <- "${h2g}"
    causal <- "${causal}"
    prop   <- "${prop}"

    ld_path  <- ${_input:r}
    out_path <- ${_output:r}
    idx      <- as.integer(str_match(basename(ld_path), "sample_([0-9]+)_")[, 2])

    cross_cs_maxcorr <- function(fit, R) {
      if (is.null(fit)) return(NA_real_)
      cs <- fit$sets$cs
      if (is.null(cs) || length(cs) < 2) return(NA_real_)
      tops <- vapply(cs, function(m) m[which.max(fit$pip[m])], integer(1))
      cc <- abs(R[tops, tops]); diag(cc) <- NA
      max(cc, na.rm = TRUE)
    }

    save_dropped <- function(reason) {
      saveRDS(list(status = "dropped", reason = reason, prop = prop,
                   index = idx, h2g = h2g, causal = causal, input_path = ld_path),
              out_path)
      cat(sprintf("[DROP] idx %d prop %s: %s\n", idx, prop, reason))
      quit(save = "no", status = 0)
    }

    ## ---- load sim output ----
    dats  <- readRDS(ld_path)
    n_UKB <- dats$n_UKB; n_ADSP <- dats$n_ADSP; N <- dats$total_n
    w     <- n_ADSP / N                        # realized natural weight (ADSP proportion)

    ## ---- subsample LD (in-sample / generative gold standard) ----
    UKB_sub <- get_cormat(dats$UKB_Xmat)
    ADSP_sub <- get_cormat(dats$ADSP_Xmat)
    if (!setequal(colnames(UKB_sub), colnames(ADSP_sub)))
      save_dropped("UKB/ADSP subsample variant sets differ")
    ref_order <- colnames(UKB_sub)
    ADSP_sub   <- ADSP_sub[ref_order, ref_order]

    ## ---- full-panel LD (realistic reference-panel case) ----
    panel    <- readRDS(dats$panel_ld_path)
    UKB_full <- panel$UKB_LD_full[ref_order, ref_order]
    ADSP_full <- panel$ADSP_LD_full[ref_order, ref_order]

    ## ---- natural-weight meta z (direct; both cohorts already aligned) ----
    e_idx <- match(ref_order, dats$UKB_target_tb$variant_id)
    a_idx <- match(ref_order, dats$ADSP_target_tb$variant_id)
     if(nrow(dats$UKB_target_tb) == 0){
         z_meta = dats$ADSP_target_tb$z
     }else if(nrow(dats$ADSP_target_tb) == 0){
         z_meta = dats$UKB_target_tb$z
     }else{
        if (anyNA(e_idx) || anyNA(a_idx)) save_dropped("sumstat variants do not cover ref_order")
        z_UKB  <- dats$UKB_target_tb$z[e_idx]
        z_ADSP  <- dats$ADSP_target_tb$z[a_idx]
        z_meta <- (sqrt(n_ADSP) * z_ADSP + sqrt(n_UKB) * z_UKB) / sqrt(N)
    }
    n_mix  <- N                                # n_eff at the natural weight = total_n


    ## ---- generative LD both ways ----
    R_gen_sub  <- w * ADSP_sub  + (1 - w) * UKB_sub     # GOLD STANDARD
    R_gen_full <- w * ADSP_full + (1 - w) * UKB_full

    fit_or_null <- function(R, tag) tryCatch(
      susie_rss(z = z_meta, R = R, n = n_mix),
      error = function(e) {
        cat(sprintf("[FAIL-FIT] idx %d prop %s %s: %s\n", idx, prop, tag, conditionMessage(e)))
        NULL })

    ## ---- fits ----
    gen_insample <- fit_or_null(R_gen_sub,  "gen_insample")   # GOLD: generative on subsample LD
    gen_refpanel <- fit_or_null(R_gen_full, "gen_refpanel")   # generative on full-panel LD
    UKB_refpanel <- fit_or_null(UKB_full,   "UKB_refpanel")   # single UKB panel
    ADSP_refpanel <- fit_or_null(ADSP_full,   "ADSP_refpanel")   # single ADSP panel
    mix_refpanel <- fit_or_null(list(ADSP_full, UKB_full), "mix_refpanel")  # omega[1]=ADSP, [2]=UKB
    om <- if (is.null(mix_refpanel)) NULL else mix_refpanel$omega_weights

    R_manual <- om[1] * ADSP_full + om[2] * UKB_full
    manual_refpanel <- fit_or_null(R_manual, "manual_refpanel")
    
    result_list <- list(
      gen_insample = gen_insample,   # <- gold-standard benchmark
      gen_refpanel = gen_refpanel,
      UKB_refpanel = UKB_refpanel,
      ADSP_refpanel = ADSP_refpanel,
      mix_refpanel = mix_refpanel,
      manual_refpanel = manual_refpanel
    )
    
    # per-fit cross-CS correlation, each against the LD it used
    R_used <- list(gen_insample = R_gen_sub, gen_refpanel = R_gen_full,
                       UKB_refpanel = UKB_full, ADSP_refpanel = ADSP_full,
                       mix_refpanel = R_gen_full,
                       manual_refpanel = if (!is.null(om) && length(om) == 2)
                                           om[1] * ADSP_full + om[2] * UKB_full else R_gen_full)
        max_cs_corr <- imap_dbl(result_list, function(fit, nm) cross_cs_maxcorr(fit, R_used[[nm]]))


    output <- list(
      status        = "ok",
      result_list   = result_list,
      n_sample      = n_mix, n_UKB = n_UKB, n_ADSP = n_ADSP, total_n = N,
      prop = prop, w = w,
      input_path    = ld_path,
      true_vars     = dats$true_vars,
      max_cs_corr   = max_cs_corr,
      omega_weights = om,
      index = idx, h2g = h2g, causal = causal
    )
    dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output, out_path)

    ncs <- function(f) if (is.null(f)) NA else length(f$sets$cs)
    cat(sprintf("[OK] idx %d prop %s  gen(in=%s ref=%s) single(UKB=%s ADSP=%s) mix=%s w=%.3f n=%.0f\n",
                idx, prop, ncs(gen_insample), ncs(gen_refpanel),
                ncs(UKB_refpanel), ncs(ADSP_refpanel), ncs(mix_refpanel), w, n_mix))


In [ ]:
#!/bin/bash
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="ADSP_UKB_fine"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/12.Fig4A_4B_S9A.ADSP_UKB_Meta.ipynb" finemap_sweep \
    --h2g H2G --causal CAUSAL --prop PROP \
    --mem 30G
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== Generate + submit one job per (causal, h2g, prop) =====
for causal in 1 2 3; do
  for h2g in 0.02 0.04 0.06; do
      for prop in 0.0 0.20 0.50 0.80 1.0; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
      sb "${output_script}"   # use sb if you aliased
    done
  done
done

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_fine/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J susie_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/log/summary.%J.out
#BSUB -e WORK_DIR/log/summary.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/14.Fig4.AFR_EUR_Meta_v1_and_Summary_Steps.ipynb" susie_mix_summary \
    --folder /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_fine/result/ \
    --mem 40G \
    --cwd WORK_DIR/summary/


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [10]:
library(plink2R)
library(tidyverse)
library(MASS)
library(crayon)
library(pecotmr)
library(susieR)
library(vroom)
library(pgenlibr)
library(Rfast)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
select <- dplyr::select

# Project utilities
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")

# -----------------------------------------------------------------------------
# Two susieR builds, each in its own environment so they never clobber.
# Each susie_*() carries its enclosing env, so internal helper lookups resolve
# against the build it came from.
#   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
#   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
# -----------------------------------------------------------------------------


# Local susieR build (multi-panel / R_finite) -- required for the list() reference
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)



h2g    <- "0.04"
causal <- "1"
prop   <- "0.80"

ld_path  <- '/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_Mix_simulation/result/h2g_0.04/causal_1/prop_0.80/sample_382_ncausal_1_h2g_0.04_meta_mix.rds'
out_path <- '/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_fine/result/h2g_0.04/causal_1/prop_0.80/sample_382_ncausal_1_h2g_0.04_meta_mix.rds'
idx      <- as.integer(str_match(basename(ld_path), "sample_([0-9]+)_")[, 2])

cross_cs_maxcorr <- function(fit, R) {
  if (is.null(fit)) return(NA_real_)
  cs <- fit$sets$cs
  if (is.null(cs) || length(cs) < 2) return(NA_real_)
  tops <- vapply(cs, function(m) m[which.max(fit$pip[m])], integer(1))
  cc <- abs(R[tops, tops]); diag(cc) <- NA
  max(cc, na.rm = TRUE)
}

save_dropped <- function(reason) {
  saveRDS(list(status = "dropped", reason = reason, prop = prop,
               index = idx, h2g = h2g, causal = causal, input_path = ld_path),
          out_path)
  cat(sprintf("[DROP] idx %d prop %s: %s\n", idx, prop, reason))
  quit(save = "no", status = 0)
}

## ---- load sim output ----
dats  <- readRDS(ld_path)
n_UKB <- dats$n_UKB; n_ADSP <- dats$n_ADSP; N <- dats$total_n
w     <- n_ADSP / N                        # realized natural weight (ADSP proportion)

## ---- subsample LD (in-sample / generative gold standard) ----
UKB_sub <- get_cormat(dats$UKB_Xmat)
ADSP_sub <- get_cormat(dats$ADSP_Xmat)
if (!setequal(colnames(UKB_sub), colnames(ADSP_sub)))
  save_dropped("UKB/ADSP subsample variant sets differ")
ref_order <- colnames(UKB_sub)
ADSP_sub   <- ADSP_sub[ref_order, ref_order]

## ---- full-panel LD (realistic reference-panel case) ----
panel    <- readRDS(dats$panel_ld_path)
UKB_full <- panel$UKB_LD_full[ref_order, ref_order]
ADSP_full <- panel$ADSP_LD_full[ref_order, ref_order]

## ---- natural-weight meta z (direct; both cohorts already aligned) ----
e_idx <- match(ref_order, dats$UKB_target_tb$variant_id)
a_idx <- match(ref_order, dats$ADSP_target_tb$variant_id)
 if(nrow(dats$UKB_target_tb) == 0){
     z_meta = dats$ADSP_target_tb$z
 }else if(nrow(dats$ADSP_target_tb) == 0){
     z_meta = dats$UKB_target_tb$z
 }else{
    if (anyNA(e_idx) || anyNA(a_idx)) save_dropped("sumstat variants do not cover ref_order")
    z_UKB  <- dats$UKB_target_tb$z[e_idx]
    z_ADSP  <- dats$ADSP_target_tb$z[a_idx]
    z_meta <- (sqrt(n_ADSP) * z_ADSP + sqrt(n_UKB) * z_UKB) / sqrt(N)
}
n_mix  <- N                                # n_eff at the natural weight = total_n


## ---- generative LD both ways ----
R_gen_sub  <- w * ADSP_sub  + (1 - w) * UKB_sub     # GOLD STANDARD
R_gen_full <- w * ADSP_full + (1 - w) * UKB_full

fit_or_null <- function(R, tag) tryCatch(
  susie_rss(z = z_meta, R = R, n = n_mix),
  error = function(e) {
    cat(sprintf("[FAIL-FIT] idx %d prop %s %s: %s\n", idx, prop, tag, conditionMessage(e)))
    NULL })

## ---- fits ----
gen_insample <- fit_or_null(R_gen_sub,  "gen_insample")   # GOLD: generative on subsample LD
gen_refpanel <- fit_or_null(R_gen_full, "gen_refpanel")   # generative on full-panel LD
UKB_refpanel <- fit_or_null(UKB_full,   "UKB_refpanel")   # single UKB panel
ADSP_refpanel <- fit_or_null(ADSP_full,   "ADSP_refpanel")   # single ADSP panel
mix_refpanel <- fit_or_null(list(ADSP_full, UKB_full), "mix_refpanel")  # omega[1]=ADSP, [2]=UKB
om <- if (is.null(mix_refpanel)) NULL else mix_refpanel$omega_weights

R_manual <- om[1] * ADSP_full + om[2] * UKB_full
manual_refpanel <- fit_or_null(R_manual, "manual_refpanel")

result_list <- list(
  gen_insample = gen_insample,   # <- gold-standard benchmark
  gen_refpanel = gen_refpanel,
  UKB_refpanel = UKB_refpanel,
  ADSP_refpanel = ADSP_refpanel,
  mix_refpanel = mix_refpanel,
  manual_refpanel = manual_refpanel
)

# per-fit cross-CS correlation, each against the LD it used
R_used <- list(gen_insample = R_gen_sub, gen_refpanel = R_gen_full,
                   UKB_refpanel = UKB_full, ADSP_refpanel = ADSP_full,
                   mix_refpanel = R_gen_full,
                   manual_refpanel = if (!is.null(om) && length(om) == 2)
                                       om[1] * ADSP_full + om[2] * UKB_full else R_gen_full)
    max_cs_corr <- imap_dbl(result_list, function(fit, nm) cross_cs_maxcorr(fit, R_used[[nm]]))


output <- list(
  status        = "ok",
  result_list   = result_list,
  n_sample      = n_mix, n_UKB = n_UKB, n_ADSP = n_ADSP, total_n = N,
  prop = prop, w = w,
  input_path    = ld_path,
  true_vars     = dats$true_vars,
  max_cs_corr   = max_cs_corr,
  omega_weights = om,
  index = idx, h2g = h2g, causal = causal
)
dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is 

In [11]:
UKB_refpanel$sets

ADSP_refpanel$sets

manual_refpanel$sets

om

,min.abs.corr,mean.abs.corr,median.abs.corr
,<dbl>,<dbl>,<dbl>
L1,0.9974657,0.9983105,0.9974772


,min.abs.corr,mean.abs.corr,median.abs.corr
,<dbl>,<dbl>,<dbl>
L1,0.9811724,0.987288,0.9813007


,min.abs.corr,mean.abs.corr,median.abs.corr
,<dbl>,<dbl>,<dbl>
L1,0.9852457,0.9900436,0.9853448


[1] 0.75 0.25